# 0.17 — Minimal unseen-edge emergence (weekly ranks)

Single function: baseline-unseen edges → micro-graph → Louvain → weekly S* ranks.

**Window:** 2022-10-01 → 2023-02-28 (weekly).

**Prerequisites:** `genai_full_meta.parquet` + `genai_graph_terms.parquet` (from 0.10 / 0.13).

## Algorithm (per week)

**Baseline** (through 2022-09-30): build the set of term pairs ever seen co-occurring in a headline.

**1. Unseen edges** — For each discovery week, count co-occurring pairs \((a,b)\). Keep pairs **not in baseline** with co-occurrence count \(c \ge 2\).

**2. Edge score** — For each unseen pair:

\[
\text{burst} = \log\frac{c + \alpha}{\bar{c}_{\text{baseline}} + \alpha}, \quad
\text{WoW} = \frac{c}{c_{\text{prev week}}}, \quad
\text{edge\_score} = \text{burst} \cdot \log(1+c) \cdot \min(\text{WoW}, \text{cap})
\]

Drop edges with WoW \(< 1.5\) unless \(c \ge 5\) (filters one-off spikes).

**3. Micro-graph** — Take top 100 edges by `edge_score`; edge weight = score. Run **Louvain** on this graph only.

**4. Communities** — Keep clusters with \(\ge 3\) terms and \(\ge 2\) internal seed edges. Attach headlines that share any community term.

**5. S\* rank (within week)** — For each community, z-score three features across communities in that week, then average:

- mean internal `edge_score`
- number of internal edges
- headline share (community headlines / week total)

\[
S^* = \frac{1}{3}\left(z_{\text{mean edge score}} + z_{\text{n edges}} + z_{\text{share}}\right)
\]

Lower `rank_in_week` = higher \(S^*\) (rank 1 = strongest emergence that week).

In [1]:
from collections import Counter, defaultdict
from itertools import combinations
from math import log
from pathlib import Path

import networkx as nx
import numpy as np
import pandas as pd
from networkx.algorithms.community import louvain_communities
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUTPUT_DIR = _ROOT / "notebooks" / "output"

BASELINE_END = pd.Timestamp("2022-09-30")
DISCOVERY_START = pd.Timestamp("2022-10-01")
DISCOVERY_END = pd.Timestamp("2023-02-28")
FREQ = "W-MON"
ALPHA = 1.0
MIN_COOC = 2
TOP_EDGES = 100
WOW_MIN = 1.5
WOW_CAP = 10.0
MIN_COMM_SIZE = 3
MIN_INTERNAL_EDGES = 2
LOUVAIN_SEED = 42

TERM_STOP = set(ENGLISH_STOP_WORDS) | {
    "says", "said", "new", "year", "week", "report", "shares", "stock",
    "jan", "feb", "mar", "apr", "may", "jun", "jul", "aug", "sep", "oct", "nov", "dec",
}


def normalize_terms(v):
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return []
    if isinstance(v, np.ndarray):
        return v.tolist()
    return list(v) if isinstance(v, (list, tuple)) else []


def filter_terms(ts):
    return [t for t in ts if len(t) >= 3 and not any(tok in TERM_STOP for tok in t.split())]


def ek(a, b):
    return (a, b) if a < b else (b, a)


def week_ts(w):
    return pd.Period(w, freq=FREQ).start_time


print(f"Discovery {DISCOVERY_START.date()} → {DISCOVERY_END.date()}")

Discovery 2022-10-01 → 2023-02-28


In [2]:
def detect_unseen_edge_communities(
    news: pd.DataFrame,
    baseline_end: pd.Timestamp,
    discovery_start: pd.Timestamp,
    discovery_end: pd.Timestamp,
) -> pd.DataFrame:
    """Minimal 0.16: unseen edges → micro-graph → Louvain → weekly S* ranks."""
    df = news.copy()
    df["terms_f"] = df["terms"].map(normalize_terms).map(filter_terms)
    df["week"] = df["date"].dt.to_period(FREQ).astype(str)

    bl = df.loc[df.date <= baseline_end]
    baseline_pairs = set()
    for tlist in bl["terms_f"]:
        baseline_pairs.update(combinations(sorted(set(tlist)), 2))

    twc = defaultdict(lambda: defaultdict(int))
    ewc = defaultdict(lambda: defaultdict(int))
    for week, grp in df.groupby("week"):
        for tlist in grp["terms_f"]:
            for t in set(tlist):
                twc[t][week] += 1
            for a, b in combinations(sorted(set(tlist)), 2):
                ewc[ek(a, b)][week] += 1

    all_weeks = sorted(df["week"].unique(), key=week_ts)
    bl_weeks = [w for w in all_weeks if week_ts(w) <= baseline_end]
    disc_weeks = [w for w in all_weeks if discovery_start <= week_ts(w) <= discovery_end]

    def bexp(counts):
        return float(np.mean([counts.get(w, 0) for w in bl_weeks])) if bl_weeks else 0.0

    rows = []
    prior_pc = Counter()

    for wi, week in enumerate(disc_weeks):
        grp = df.loc[df.week == week]
        pc = Counter()
        for tlist in grp["terms_f"]:
            pc.update(combinations(sorted(set(tlist)), 2))

        if wi > 0:
            prior_pc = Counter()
            for tlist in df.loc[df.week == disc_weeks[wi - 1], "terms_f"]:
                prior_pc.update(combinations(sorted(set(tlist)), 2))

        edges = []
        for (a, b), cooc in pc.items():
            if ek(a, b) in baseline_pairs or cooc < MIN_COOC:
                continue
            eb = log((cooc + ALPHA) / (bexp(ewc[ek(a, b)]) + ALPHA))
            wow = cooc / max(prior_pc.get((a, b), 0), 1)
            if wow < WOW_MIN and cooc < 5:
                continue
            score = eb * log(1 + cooc) * min(wow, WOW_CAP)
            edges.append((score, a, b))

        edges.sort(reverse=True)
        g = nx.Graph()
        for score, a, b in edges[:TOP_EDGES]:
            g.add_edge(a, b, weight=score)

        if g.number_of_edges() < 2:
            continue

        week_total = len(grp)
        comms = []
        for cid, term_set in enumerate(louvain_communities(g, weight="weight", seed=LOUVAIN_SEED)):
            terms = set(term_set)
            if len(terms) < MIN_COMM_SIZE:
                continue
            internal = [(a, b, g[a][b]["weight"]) for a, b in combinations(sorted(terms), 2) if g.has_edge(a, b)]
            if len(internal) < MIN_INTERNAL_EDGES:
                continue
            members = sum(1 for tlist in grp["terms_f"] if terms & set(tlist))
            top_terms = sorted(terms, key=lambda t: twc[t].get(week, 0), reverse=True)[:10]
            comms.append({
                "community_id": cid,
                "mean_edge_score": float(np.mean([x[2] for x in internal])),
                "n_edges": len(internal),
                "headline_count": members,
                "share": members / week_total if week_total else 0,
                "top_terms": ", ".join(top_terms),
            })

        if not comms:
            continue

        cdf = pd.DataFrame(comms)
        for col in ["mean_edge_score", "n_edges", "share"]:
            std = cdf[col].std(ddof=0)
            cdf[f"z_{col}"] = (cdf[col] - cdf[col].mean()) / std if std else 0.0
        cdf["S_star"] = cdf[["z_mean_edge_score", "z_n_edges", "z_share"]].mean(axis=1)
        cdf["rank_in_week"] = cdf["S_star"].rank(ascending=False, method="first").astype(int)
        cdf["week"] = week
        rows.append(cdf)

    out_cols = ["week", "community_id", "rank_in_week", "S_star", "headline_count", "share", "n_edges", "top_terms"]
    return pd.concat(rows, ignore_index=True)[out_cols] if rows else pd.DataFrame(columns=out_cols)

In [3]:
meta = pd.read_parquet(OUTPUT_DIR / "genai_full_meta.parquet")
terms = pd.read_parquet(OUTPUT_DIR / "genai_graph_terms.parquet")
meta["date"] = pd.to_datetime(meta["date"]).dt.normalize()
terms["date"] = pd.to_datetime(terms["date"]).dt.normalize()
terms = terms.drop_duplicates(["Headline", "date"], keep="first")

news = meta.merge(terms[["Headline", "date", "terms"]], on=["Headline", "date"], how="left")
news = news.loc[news.date <= DISCOVERY_END]

rankings = detect_unseen_edge_communities(news, BASELINE_END, DISCOVERY_START, DISCOVERY_END)
print(f"{len(rankings):,} community-weeks across {rankings.week.nunique()} weeks")

413 community-weeks across 22 weeks


In [4]:
cols = ["rank_in_week", "S_star", "headline_count", "share", "top_terms"]
for week in sorted(rankings.week.unique(), key=week_ts):
    sub = rankings.loc[rankings.week == week].sort_values("rank_in_week")
    print(f"\n{'=' * 72}\n{week}  ({len(sub)} communities)\n{'=' * 72}")
    print(sub[cols].to_string(index=False, float_format=lambda x: f"{x:.3f}"))


2022-10-04/2022-10-10  (20 communities)
 rank_in_week  S_star  headline_count  share                                                                                              top_terms
            1   2.080             508  0.035 billion, twitter, close, bid, imperial brands, revives, musk revives, lurches, twitter lurches, linger
            2   1.084             263  0.018                             credit, suisse, credit suisse, hsbc, draws, pimco, spg, billion debt, chua
            3   0.605             251  0.017                                         talks, health, cvs, medicare, cano, exclusive, exclusive talks
            4   0.407              43  0.003                                                                coast, hurricane, nicaragua, julia, nhc
            5   0.400             109  0.008                                                                   covered, long, crh, edp, edp finance
            6   0.311              78  0.005                           

In [5]:
import re

HINT = re.compile(r"chatgpt|openai|chatbot|generative|gpt", re.I)
hits = rankings.loc[rankings.top_terms.str.contains(HINT, na=False)].sort_values(["week", "rank_in_week"])
if hits.empty:
    print("No genAI-hint communities found.")
else:
    print(f"GenAI-hint communities ({len(hits)} rows):")
    print(hits[["week", "rank_in_week", "S_star", "top_terms"]].to_string(index=False, float_format=lambda x: f"{x:.3f}"))

GenAI-hint communities (1 rows):
                 week  rank_in_week  S_star                                                                              top_terms
2023-01-17/2023-01-23             1   1.813 cuts, investment, job, microsoft, maker, openai, chatgpt, chatgpt maker, microsoft job
